# Controlled Decoding (Mudgal et al., ICML 2024): шаг 1, роллауты

Воспроизводим Exp. 2 (Anthropic HH) с открытыми моделями: блочный CD-FUDGE против best-of-K.

| В статье | Здесь |
|---|---|
| base π_ref: PaLM 2-XXS | `Qwen/Qwen2.5-0.5B-Instruct` |
| reward: PaLM 2-XXS, обученный на HH | `Skywork/Skywork-Reward-V2-Qwen3-0.6B` |

Этот ноутбук:
1. Берёт промпты из `Anthropic/hh-rlhf`: train нужен для обучения prefix scorer'а, test для оценки.
2. Для каждого train-промпта сэмплирует `N_SAMPLES` ответов из π_ref на `cuda:0`.
3. Оценивает ответы Skywork RM на `cuda:1`.
4. Сохраняет `rollouts/train.parquet`, `rollouts/test_prompts.parquet` и `rollouts/config.json`.

CD-FUDGE учит V(x, yᵗ) предсказывать итоговый reward r(x, y) по каждому префиксу yᵗ. Поэтому сохраняем
токены ответа (`response_ids`), а не только текст: префиксы потом должны совпадать с тем, что
сэмплирует π_ref.

**Важно:** π_ref задаётся сэмплированием с `temperature=1`, без top-p, top-k и repetition penalty.
У Qwen в `generation_config` по умолчанию стоят top_p=0.8, top_k=20 и repetition_penalty=1.05,
поэтому все параметры передаём явно. В `03_eval` нужно использовать тот же `GEN_KW`, его берём из `config.json`.

**Kaggle:** Accelerator = GPU T4 x2, Internet = On. Обработка идёт шардами, поэтому если сессия упадёт,
повторный запуск продолжит с места остановки. После полного прогона: Save Version → Output → New Dataset,
этот датасет подключается во втором ноутбуке.

In [ ]:
# Обычно transformers на Kaggle уже свежий, и эта ячейка не нужна.
# Если всё же ставишь пакеты: запусти ячейку, затем Run → Restart & Clear Cell Outputs и выполни ноутбук сверху,
# не запуская её повторно. Иначе в ядре смешаются старая и новая версии (ImportError из transformers.utils.import_utils).
# !pip install -q -U "transformers>=4.56" accelerate

In [ ]:
import os, re, json, glob, time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification

print(torch.__version__, torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
DEBUG = True  # сначала короткий прогон на нескольких промптах; для полного прогона поставить False

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
RM_MODEL = "Skywork/Skywork-Reward-V2-Qwen3-0.6B"

N_TRAIN = 64 if DEBUG else 10_000  # промпты для обучения scorer'а
N_TEST = 32 if DEBUG else 500      # промпты для оценки (генерация в 03_eval)
N_SAMPLES = 2                      # ответов π_ref на один train-промпт
MAX_PROMPT_TOKENS = 512
MAX_NEW_TOKENS = 256  # при 128 почти половина ответов обрезалась
SHARD_SIZE = 32 if DEBUG else 1000
GEN_BATCH = 64                     # промптов за один вызов generate (каждый даёт N_SAMPLES последовательностей)
RM_BATCH = 32
SEED = 0

GEN_KW = dict(do_sample=True, temperature=1.0, top_p=1.0, top_k=0, repetition_penalty=1.0, max_new_tokens=MAX_NEW_TOKENS)

GEN_DEVICE = "cuda:0"
RM_DEVICE = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
DTYPE = torch.float16  # T4 не поддерживает bf16

OUT = ("/kaggle/working/" if os.path.exists("/kaggle") else "") + ("rollouts_debug" if DEBUG else "rollouts")  # иначе полный прогон подхватит отладочные шарды
SHARDS = f"{OUT}/shards"
os.makedirs(SHARDS, exist_ok=True)

# Шарды переиспользуются только при тех же параметрах: иначе после перезапуска подхватятся чужие данные
RUN_META = {"base_model": BASE_MODEL, "rm_model": RM_MODEL, "n_train": N_TRAIN, "n_samples": N_SAMPLES,
            "max_prompt_tokens": MAX_PROMPT_TOKENS, "shard_size": SHARD_SIZE, "gen_kw": GEN_KW, "seed": SEED}
meta_path = f"{SHARDS}/run_meta.json"
if os.path.exists(meta_path):
    old = json.load(open(meta_path))
    assert old == RUN_META, f"В {SHARDS} лежат шарды с другими параметрами: {old}. Удали папку: !rm -rf {SHARDS}"
elif glob.glob(f"{SHARDS}/*.parquet"):
    raise RuntimeError(f"В {SHARDS} лежат шарды неизвестного прогона. Удали папку: !rm -rf {SHARDS}")
else:
    json.dump(RUN_META, open(meta_path, "w"), indent=2)

## Промпты

Текст в hh-rlhf выглядит так: `\n\nHuman: ...\n\nAssistant: ...` (несколько ходов). Промпт — это диалог
без последней реплики Assistant. Его переводим в формат messages и применяем chat template Qwen.

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.padding_side = "left"

def parse_hh(text):
    parts = re.split(r"\n\n(Human|Assistant): ", "\n\n" + text.strip())[1:]
    msgs = [{"role": "user" if r == "Human" else "assistant", "content": c.strip()} for r, c in zip(parts[::2], parts[1::2])]
    while msgs and msgs[-1]["role"] == "assistant":
        msgs.pop()
    return msgs

def build_prompts(split, n, seed):
    ds = load_dataset("Anthropic/hh-rlhf", split=split).shuffle(seed=seed)
    rows, seen = [], set()
    for ex in ds:
        msgs = parse_hh(ex["chosen"])
        if not msgs or any(not m["content"] for m in msgs):
            continue
        key = json.dumps(msgs)
        if key in seen:
            continue
        ids = tok(tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False), add_special_tokens=False).input_ids
        if len(ids) > MAX_PROMPT_TOKENS:
            continue
        seen.add(key)
        rows.append({"pid": len(rows), "messages": key, "prompt_ids": ids})
        if len(rows) == n:
            break
    return pd.DataFrame(rows)

train_prompts = build_prompts("train", N_TRAIN, SEED)
test_prompts = build_prompts("test", N_TEST, SEED)
test_prompts.to_parquet(f"{OUT}/test_prompts.parquet")
print(len(train_prompts), len(test_prompts))
print(tok.decode(train_prompts.prompt_ids[0]))

## Генерация из π_ref

Ответ обрезаем по первому EOS и сам EOS оставляем: для value function терминальный токен
тоже является шагом. `finished=False` означает, что ответ упёрся в `MAX_NEW_TOKENS`. RM оценивает и такие обрезанные ответы.

In [ ]:
policy = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=DTYPE).to(GEN_DEVICE).eval()
EOS_IDS = set(policy.generation_config.eos_token_id if isinstance(policy.generation_config.eos_token_id, list) else [policy.generation_config.eos_token_id])
print("eos:", EOS_IDS, "pad:", tok.pad_token_id)

def cut_at_eos(ids):
    for i, t in enumerate(ids):
        if t in EOS_IDS:
            return ids[: i + 1], True
    return ids, False

@torch.no_grad()
def generate(prompt_ids_list):
    batch = tok.pad({"input_ids": prompt_ids_list}, return_tensors="pt").to(GEN_DEVICE)
    out = policy.generate(**batch, **GEN_KW, num_return_sequences=N_SAMPLES, pad_token_id=tok.pad_token_id)
    out = out[:, batch["input_ids"].shape[1]:].tolist()
    return [cut_at_eos(o) for o in out]

def shard_path(i, stage):
    return f"{SHARDS}/{stage}_{i:03d}.parquet"

torch.manual_seed(SEED)
n_shards = (len(train_prompts) + SHARD_SIZE - 1) // SHARD_SIZE
done = sum(len(train_prompts.iloc[s * SHARD_SIZE:(s + 1) * SHARD_SIZE]) for s in range(n_shards) if os.path.exists(shard_path(s, "gen")))
pbar = tqdm(total=len(train_prompts), initial=done, desc="generate", unit="prompt")
for s in range(n_shards):
    if os.path.exists(shard_path(s, "gen")):
        continue
    pbar.set_postfix(shard=f"{s + 1}/{n_shards}")
    t0 = time.time()
    # сортировка по длине уменьшает паддинг
    shard = train_prompts.iloc[s * SHARD_SIZE:(s + 1) * SHARD_SIZE].sort_values("prompt_ids", key=lambda c: c.map(len))
    rows = []
    for b in range(0, len(shard), GEN_BATCH):
        chunk = shard.iloc[b:b + GEN_BATCH]
        res = generate(chunk.prompt_ids.tolist())
        for j, (_, p) in enumerate(chunk.iterrows()):
            for k in range(N_SAMPLES):
                ids, fin = res[j * N_SAMPLES + k]
                rows.append({"pid": p.pid, "sample": k, "messages": p.messages, "prompt_ids": p.prompt_ids,
                             "response_ids": ids, "finished": fin,
                             "response": tok.decode(ids, skip_special_tokens=True).strip()})
        pbar.update(len(chunk))
    pd.DataFrame(rows).to_parquet(shard_path(s, "gen"))
    pbar.write(f"gen shard {s + 1}/{n_shards}: {time.time() - t0:.0f}s")
pbar.close()

del policy
torch.cuda.empty_cache()

## Reward: Skywork-Reward-V2

Используем как в model card: chat template RM на диалог `[..., assistant: response]`, скаляр берём
из `logits[:, 0]`. Модель рекомендуют запускать в bf16, у нас fp16, поэтому сначала сверяем
fp16 с fp32 на нескольких примерах.

In [ ]:
rm_tok = AutoTokenizer.from_pretrained(RM_MODEL)
rm_tok.padding_side = "right"

def rm_texts(df):
    texts = []
    for m, r in zip(df.messages, df.response):
        t = rm_tok.apply_chat_template(json.loads(m) + [{"role": "assistant", "content": r}], tokenize=False)
        if rm_tok.bos_token and t.startswith(rm_tok.bos_token):
            t = t[len(rm_tok.bos_token):]
        texts.append(t)
    return texts

@torch.no_grad()
def score(rm, texts, device, bs=RM_BATCH, pbar=None):
    scores = []
    for b in range(0, len(texts), bs):
        enc = rm_tok(texts[b:b + bs], return_tensors="pt", padding=True).to(device)
        scores += rm(**enc).logits[:, 0].float().tolist()
        if pbar is not None:
            pbar.update(len(enc["input_ids"]))
    return np.array(scores)

def load_rm(dtype):
    rm = AutoModelForSequenceClassification.from_pretrained(RM_MODEL, dtype=dtype, num_labels=1).to(RM_DEVICE).eval()
    rm.config.pad_token_id = rm_tok.pad_token_id
    return rm

probe = rm_texts(pd.read_parquet(shard_path(0, "gen")).head(16))
rm32 = load_rm(torch.float32)
s32 = score(rm32, probe, RM_DEVICE, bs=4)
del rm32
torch.cuda.empty_cache()
rm = load_rm(DTYPE)
s16 = score(rm, probe, RM_DEVICE, bs=4)
print("fp32:", s32.round(2))
print("fp16:", s16.round(2))
print("max |diff|:", np.abs(s32 - s16).max(), "| corr:", np.corrcoef(s32, s16)[0, 1])
assert np.isfinite(s16).all(), "fp16 дал NaN/inf: поставь DTYPE = torch.float32 для RM"

In [ ]:
done = N_SAMPLES * sum(len(train_prompts.iloc[s * SHARD_SIZE:(s + 1) * SHARD_SIZE]) for s in range(n_shards) if os.path.exists(shard_path(s, "scored")))
pbar = tqdm(total=N_SAMPLES * len(train_prompts), initial=done, desc="reward", unit="resp")
for s in range(n_shards):
    if os.path.exists(shard_path(s, "scored")):
        continue
    pbar.set_postfix(shard=f"{s + 1}/{n_shards}")
    t0 = time.time()
    df = pd.read_parquet(shard_path(s, "gen"))
    df["reward"] = score(rm, rm_texts(df), RM_DEVICE, pbar=pbar)
    assert np.isfinite(df.reward).all()
    df.to_parquet(shard_path(s, "scored"))
    pbar.write(f"rm shard {s + 1}/{n_shards}: {time.time() - t0:.0f}s")
pbar.close()

In [ ]:
train = pd.concat([pd.read_parquet(p) for p in sorted(glob.glob(f"{SHARDS}/scored_*.parquet"))], ignore_index=True)
train.to_parquet(f"{OUT}/train.parquet")
json.dump({"base_model": BASE_MODEL, "rm_model": RM_MODEL, "gen_kw": GEN_KW, "n_samples": N_SAMPLES,
           "max_prompt_tokens": MAX_PROMPT_TOKENS, "seed": SEED, "n_train_prompts": len(train_prompts),
           "n_test_prompts": len(test_prompts), "debug": DEBUG}, open(f"{OUT}/config.json", "w"), indent=2)

## Sanity checks

На что смотреть:
- распределение reward: если разброс маленький, scorer'у нечего учить;
- доля `finished`: если заметная доля ответов обрезана по `MAX_NEW_TOKENS`, стоит поднять `MAX_NEW_TOKENS`;
- корреляция reward с длиной: если она сильная, RM в основном награждает длину (в статье это отдельный Exp. 4);
- разброс reward между двумя сэмплами одного промпта: если внутри промпта разброс заметно меньше, чем
  между промптами, V в основном выучит «насколько хорош промпт», и выбор блоков будет слабым.

In [ ]:
import matplotlib.pyplot as plt

train["resp_len"] = train.response_ids.map(len)
print(train[["reward", "resp_len"]].describe().round(2))
print("finished:", train.finished.mean().round(3))
print("mean reward | finished:", train.groupby("finished").reward.mean().round(2).to_dict())
print("corr(reward, len):", np.corrcoef(train.reward, train.resp_len)[0, 1].round(3))
within = train.groupby("pid").reward.std().mean()
print(f"std reward: total {train.reward.std():.2f}, внутри промпта {within:.2f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].hist(train.reward, bins=50); ax[0].set_title("reward")
ax[1].scatter(train.resp_len, train.reward, s=3, alpha=0.3); ax[1].set_xlabel("response tokens"); ax[1].set_ylabel("reward")
plt.show()

In [ ]:
for _, r in train.sort_values("reward").iloc[[0, len(train) // 2, -1]].iterrows():
    print(f"=== reward {r.reward:.2f} ===")
    print("USER:", json.loads(r.messages)[-1]["content"][:300])
    print("ASSISTANT:", r.response[:500], "\n")